In [2]:
!pip uninstall -y torch torchvision torchaudio datasets transformers accelerate
!pip install -q torch torchvision torchaudio
!pip install -q transformers==4.55.0 datasets==4.0.0 accelerate evaluate sentencepiece

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 M

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate

import joblib

In [2]:
import torch
import transformers
import datasets

print(torch.__version__)
print(transformers.__version__)
print(datasets.__version__)
print(torch.cuda.is_available())

2.13.0+cu130
4.55.0
4.0.0
True


In [3]:
from google.colab import files
uploaded = files.upload()

Saving final_emotion_dataset.csv to final_emotion_dataset.csv


In [7]:
df = pd.read_csv("final_emotion_dataset.csv")

In [5]:
df = df[["clean_text","main_emotion"]]
df.dropna(inplace=True)

df.head()

,clean_text,main_emotion
0,experienced emotion grandfather passed away,Sad
1,first moved walked everywhere within week purs...,Neutral
2,oh bleated voice high rather indignant,Angry
3,however right hon gentleman recognise profound...,Fear
4,boyfriend not turn promising coming,Sad


In [6]:
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["main_emotion"])
joblib.dump(
    encoder,
    "/content/main_emotion_encoder.pkl"
)

['/content/main_emotion_encoder.pkl']

In [36]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["clean_text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [37]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [38]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=False,
    max_length=256
)

In [41]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer)

In [42]:
import torch
from torch.utils.data import Dataset

class EmotionDataset(Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

In [43]:
import torch
from torch.utils.data import Dataset

class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels
    def __len__(self):

        return len(self.labels)
    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

In [44]:
train_dataset = EmotionDataset(
    train_encodings,
    train_labels
)

test_dataset = EmotionDataset(
    test_encodings,
    test_labels
)

In [45]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(encoder.classes_)
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [46]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [47]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

In [48]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./emotion_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.05,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)

In [49]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [51]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.180100,1.270837,0.539691,0.525396
2,1.188100,1.273481,0.541589,0.518674
3,1.104200,1.310389,0.532150,0.515880
4,1.074100,1.347728,0.526906,0.513923


TrainOutput(global_step=40048, training_loss=1.1296298276220758, metrics={'train_runtime': 2893.7713, 'train_samples_per_second': 221.425, 'train_steps_per_second': 13.839, 'total_flos': 2.1230815588798464e+16, 'train_loss': 1.1296298276220758, 'epoch': 4.0})

In [52]:
results = trainer.evaluate()
print(results)

{'eval_loss': 1.2708371877670288, 'eval_accuracy': 0.539690863235698, 'eval_f1': 0.5253959208141963, 'eval_runtime': 42.5989, 'eval_samples_per_second': 940.095, 'eval_steps_per_second': 58.757, 'epoch': 4.0}


In [53]:
model.save_pretrained("/content/distilbert_emotion")
tokenizer.save_pretrained("/content/distilbert_emotion")

('/content/distilbert_emotion/tokenizer_config.json',
 '/content/distilbert_emotion/special_tokens_map.json',
 '/content/distilbert_emotion/vocab.txt',
 '/content/distilbert_emotion/added_tokens.json',
 '/content/distilbert_emotion/tokenizer.json')

In [54]:
!zip -r distilbert_emotion.zip /content/distilbert_emotion

  adding: content/distilbert_emotion/ (stored 0%)
  adding: content/distilbert_emotion/special_tokens_map.json (deflated 42%)
  adding: content/distilbert_emotion/vocab.txt (deflated 53%)
  adding: content/distilbert_emotion/model.safetensors (deflated 8%)
  adding: content/distilbert_emotion/config.json (deflated 55%)
  adding: content/distilbert_emotion/tokenizer_config.json (deflated 75%)
  adding: content/distilbert_emotion/tokenizer.json (deflated 71%)


In [61]:
from google.colab import files
files.download("distilbert_emotion.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
print(model.config.num_labels)
print(model.config.id2label)

9
{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5', 6: 'LABEL_6', 7: 'LABEL_7', 8: 'LABEL_8'}


In [60]:
print(encoder.classes_)

['Affection' 'Angry' 'Curiosity' 'Embarrassment' 'Fear' 'Happy' 'Neutral'
 'Relief' 'Sad']
